# 1 — Download Your Data

**Project:** What predicts life expectancy at the US county level?

**Y (outcome):** Life expectancy at birth (years), county-level. Source: [County Health Rankings & Roadmaps](https://www.countyhealthrankings.org/) 2024 analytic data file. CHR life-expectancy estimates originate with the National Center for Health Statistics.


In [1]:
import pandas as pd
import numpy as np
import requests
import io
import os

# County Health Rankings publishes one analytic data CSV per year.
# 2024 file URL pattern (stable across recent years):
CHR_URL = 'https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2024.csv'

print('Fetching:', CHR_URL)
r = requests.get(CHR_URL, timeout=60)
r.raise_for_status()
print('Got', len(r.content) // 1024, 'KB')

Fetching: https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2024.csv
Got 12493 KB


CHR's analytic data file uses a two-row header in some years: row 0 is the descriptive name (e.g., `Life Expectancy raw value`), row 1 is the measure code (e.g., `v147_rawvalue`). Read it as a regular CSV and the column names you'll see are the descriptive ones.

In [2]:

raw = pd.read_csv(io.BytesIO(r.content), low_memory=False)
print('Shape:', raw.shape)
print('First 5 columns:', list(raw.columns[:5]))

Shape: (3196, 770)
First 5 columns: ['State FIPS Code', 'County FIPS Code', '5-digit FIPS Code', 'State Abbreviation', 'Name']


In [3]:
#  do a fuzzy search to find the life expectancy column, as can change year to year
candidates = [c for c in raw.columns if 'life expectancy' in c.lower() and 'raw' in c.lower()]
print('Life expectancy column candidates:', candidates)
assert len(candidates) >= 1, 'No life-expectancy column found - inspect raw.columns and pick manually.'
LIFE_COL = candidates[0]
print('Using:', LIFE_COL)

Life expectancy column candidates: ['Life Expectancy raw value']
Using: Life Expectancy raw value


In [4]:
# CHR includes state-level and US-summary rows. We want county rows only.
# County rows have a 5-digit FIPS = state FIPS (2 digits) + county FIPS (3 digits).
# CHR usually has separate 'statecode' and 'countycode' (or '5-digit FIPS Code') columns.
fips_candidates = [c for c in raw.columns if 'fips' in c.lower() or '5-digit' in c.lower()]
print('FIPS-related columns:', fips_candidates)

FIPS-related columns: ['State FIPS Code', 'County FIPS Code', '5-digit FIPS Code']


In [5]:
# Try to find a single 5-digit FIPS column; otherwise build it from state + county.
five_digit = [c for c in raw.columns if '5-digit' in c.lower() and 'fips' in c.lower()]
if five_digit:
    raw['fips'] = raw[five_digit[0]].astype(str).str.zfill(5)
    print('Using 5-digit FIPS column directly')
else:
    # Build from state + county codes
    state_col  = [c for c in raw.columns if c.lower() in ('statecode','state fips code','state')][0]
    county_col = [c for c in raw.columns if c.lower() in ('countycode','county fips code','county')][0]
    raw['fips'] = (raw[state_col].astype(str).str.zfill(2)
                 + raw[county_col].astype(str).str.zfill(3))
    print(f'Built FIPS from {state_col} + {county_col}')

# Filter to county-level rows: state FIPS != '00' and county FIPS != '000'
raw['state_fips']  = raw['fips'].str[:2]
raw['county_fips'] = raw['fips'].str[2:]
county_rows = raw[(raw['state_fips'] != '00') & (raw['county_fips'] != '000')].copy()
print('County rows:', len(county_rows))

Using 5-digit FIPS column directly
County rows: 3144


In [6]:
# Find a county-name column (varies year to year).
name_candidates = [c for c in county_rows.columns if c.lower() in ('county','name','countyname')]
name_col = name_candidates[0] if name_candidates else None
state_name_candidates = [c for c in county_rows.columns if c.lower() in ('state','statename','state name')]
state_name_col = state_name_candidates[0] if state_name_candidates else None
print('County name col:', name_col, '| State name col:', state_name_col)

County name col: Name | State name col: None


In [11]:
# CHR 2024 doesn't include a text state name — build one from the state FIPS code
STATE_FIPS_TO_NAME = {
    '01': 'Alabama', '02': 'Alaska', '04': 'Arizona', '05': 'Arkansas',
    '06': 'California', '08': 'Colorado', '09': 'Connecticut', '10': 'Delaware',
    '11': 'District of Columbia', '12': 'Florida', '13': 'Georgia', '15': 'Hawaii',
    '16': 'Idaho', '17': 'Illinois', '18': 'Indiana', '19': 'Iowa',
    '20': 'Kansas', '21': 'Kentucky', '22': 'Louisiana', '23': 'Maine',
    '24': 'Maryland', '25': 'Massachusetts', '26': 'Michigan', '27': 'Minnesota',
    '28': 'Mississippi', '29': 'Missouri', '30': 'Montana', '31': 'Nebraska',
    '32': 'Nevada', '33': 'New Hampshire', '34': 'New Jersey', '35': 'New Mexico',
    '36': 'New York', '37': 'North Carolina', '38': 'North Dakota', '39': 'Ohio',
    '40': 'Oklahoma', '41': 'Oregon', '42': 'Pennsylvania', '44': 'Rhode Island',
    '45': 'South Carolina', '46': 'South Dakota', '47': 'Tennessee', '48': 'Texas',
    '49': 'Utah', '50': 'Vermont', '51': 'Virginia', '53': 'Washington',
    '54': 'West Virginia', '55': 'Wisconsin', '56': 'Wyoming',
}
county_rows['state_name'] = county_rows['state_fips'].map(STATE_FIPS_TO_NAME)

# Build the clean outcome dataframe
keep_cols = {
    'fips':            'fips',
    'Name':            'county_name',
    'state_name':      'state_name',
    LIFE_COL:          'life_expectancy',
}
outcomes = county_rows[list(keep_cols)].rename(columns=keep_cols).copy()
outcomes['life_expectancy'] = pd.to_numeric(outcomes['life_expectancy'], errors='coerce')
outcomes = outcomes.dropna(subset=['life_expectancy']).reset_index(drop=True)

# Sanity check — should have ~3,100 rows and no missing state names
print('Rows:', len(outcomes))
print('Missing state_name:', outcomes['state_name'].isna().sum())
outcomes.head()

Rows: 3070
Missing state_name: 0


,fips,county_name,state_name,life_expectancy
0,01001,Autauga County,Alabama,75.263497
1,01003,Baldwin County,Alabama,76.738314
2,01005,Barbour County,Alabama,72.377024
3,01007,Bibb County,Alabama,72.251369
4,01009,Blount County,Alabama,73.376568


In [12]:

print(outcomes['life_expectancy'].describe().round(2))

count    3070.00
mean       75.75
std         3.43
min        55.92
25%        73.57
50%        75.84
75%        77.99
max        98.90
Name: life_expectancy, dtype: float64


In [13]:
os.makedirs('data', exist_ok=True)
outcomes.to_csv('data/county_outcomes.csv', index=False)
print('Wrote data/county_outcomes.csv -', len(outcomes), 'counties')

Wrote data/county_outcomes.csv - 3070 counties
